# Exploração e tratamento inicial de dados com Pandas

Este notebook e um guia de bolso para receber uma nova base tabular, entender sua estrutura, padronizá-la e iniciar análises com segurança.

A sequencia importa: primeiro conheça os dados; depois trate problemas; por fim, crie métricas e responda perguntas.


## Roteiro reutilizável

1. Preparar o ambiente e carregar a fonte.
2. Diagnosticar estrutura, qualidade e cobertura dos dados.
3. Padronizar nomes e tipos de acordo com o significado de cada coluna.
4. Criar variáveis derivadas com comparações, fórmulas, funções e `apply()`.
5. Fazer recortes e validar os resultados.

Antes de alterar qualquer dado, responda: quais campos são identificadores, datas, categorias, métricas ou texto livre?


## 1. Preparação do ambiente

Importe o Pandas e registre a versão usada para tornar a análise reproduzível.


In [ ]:
import pandas as pd

print(pd.__version__)

## 2. Carregamento da base

Defina o caminho de forma relativa à localização do notebook. O backend `pyarrow` usa tipos anuláveis, pode reduzir o consumo de memória e melhora operações em bases maiores, principalmente com texto. Também facilita a interoperabilidade com Parquet, DuckDB e Polars.

Código postal é um identificador, não uma medida. Carregá-lo como texto preserva zeros à esquerda.


In [ ]:
data = "../data/superstore.csv"

df = pd.read_csv(
    data,
    encoding="ISO-8859-1",
    dtype_backend="pyarrow",
    dtype={"Codigo Postal": "string[pyarrow]"}
)

df.head()

## 3. Diagnóstico inicial

Verifique tamanho, nomes, tipos e valores ausentes antes de tomar decisões. Um tipo inferido pelo Pandas pode não corresponder ao significado de negócio da coluna.


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.describe(include="all").T

### Qualidade dos dados

A contagem de nulos ajuda a decidir se é necessário preencher, excluir ou investigar registros. Duplicidade deve ser analisada antes de qualquer remoção: linhas iguais podem ser erro ou eventos válidos.


In [ ]:
df.isna().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.loc[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

## 4. Padronização de nomes

Use `snake_case` e nomes sem espaços ou caracteres especiais. Isso deixa filtros, fórmulas e reaproveitamento do código mais simples.


In [ ]:
mapa_colunas = {
    "Linha ID": "linha_id",
    "Ordem ID": "ordem_id",
    "Data Ordem": "data_ordem",    
    "Data Envio": "data_envio",
    "Modo Envio": "modo_envio",
    "Cliente ID": "cliente_id",
    "Nome Cliente": "nome_cliente",
    "Segmento": "segmento",
    "Cidade": "cidade",
    "Estado": "estado",
    "Codigo Postal": "cod_postal",
    "Regiao": "regiao",
    "Produto ID": "produto_id",
    "Categoria": "categoria_prod",
    "Sub-Categoria": "sub_categoria_prod",
    "Nome Produto": "nome_prod",
    "Vendas": "vendas_prod",
    "Quantidade": "quantidade_vendida",
    "Desconto": "desconto_concedido",
    "Lucro": "lucro_prod",
}

# Localiza dinamicamente a coluna de País para evitar quebras por encoding do arquivo
col_pais = [c for c in df.columns if "Pa" in c]
if col_pais:
    mapa_colunas[col_pais[0]] = "pais"

df = df.rename(columns=mapa_colunas)

df.columns

## 5. Tipos de dados

Defina o tipo pelo significado do campo: datas viram `datetime`; atributos com poucas opções repetidas viram `category`; identificadores e texto livre permanecem como `string[pyarrow]`.

`astype()` altera o armazenamento, enquanto `round()` altera valores. Por isso, datas e arredondamentos são tratados em comandos próprios.


In [ ]:
for coluna in ["data_ordem", "data_envio"]:
    df[coluna] = pd.to_datetime(
        df[coluna],
        format="%m/%d/%Y",
        errors="raise"
    )

In [ ]:
df = df.astype({
    "ordem_id": "string[pyarrow]",
    "modo_envio": "category",
    "cliente_id": "string[pyarrow]",
    "nome_cliente": "string[pyarrow]",
    "segmento": "category",
    "pais": "category",
    "cidade": "string[pyarrow]",
    "estado": "category",
    "cod_postal": "string[pyarrow]",
    "regiao": "category",
    "produto_id": "string[pyarrow]",
    "categoria_prod": "category",
    "sub_categoria_prod": "category",
    "nome_prod": "string[pyarrow]",
    "vendas_prod": "float64[pyarrow]",
    "lucro_prod": "float64[pyarrow]",
})

df[["vendas_prod", "lucro_prod"]] = df[["vendas_prod", "lucro_prod"]].round(2)

df.dtypes

## 6. Criação de variáveis

Os exemplos abaixo praticam comparacoes, formulas, funcoes e `apply()`. A atribuicao `df['nova_coluna'] = ...` armazena o resultado no DataFrame. Execute na ordem, depois de padronizar os nomes das colunas.

### 6.1. Desconto aplicado e lucro por unidade

A comparação `> 0` indica se houve desconto. O lucro por unidade divide o lucro pela quantidade vendida; `round(2)` arredonda esse resultado. Verifique quantidades iguais a zero antes de reutilizar a fórmula em outra base.


In [ ]:
df['desconto_aplicado'] = df['desconto_concedido'] > 0

df['lucro_por_unidade'] = df['lucro_prod'] / df['quantidade_vendida']
df['lucro_por_unidade'] = df['lucro_por_unidade'].round(2)

### 6.2. Funcao para categorizar o lucro

`apply(categorizar_lucro)` chama a funcao para cada valor de `lucro_prod`. Acima de 50 retorna 'Margem Alta'; acima de zero até 50 retorna 'Margem Média'; zero e valores negativos retornam 'Margem Negativa'. Esses rótulos preservam a regra do exercício: classificam o lucro absoluto, não a margem percentual.


In [ ]:
def categorizar_lucro(lucro_prod):
    if lucro_prod > 50:
        return 'Margem Alta'
    elif lucro_prod > 0:
        return 'Margem Média'
    else:
        return 'Margem Negativa'

df['categoria_lucro'] = df['lucro_prod'].apply(categorizar_lucro)

### 6.3. Margem de lucro percentual (apply vs vetorização)

`lambda` define uma função anônima. Com `axis=1`, o `apply()` percorre linha a linha: `x` representa uma linha e permite acessar lucro e vendas juntos. A fórmula divide lucro por vendas e multiplica por 100.

> 💡 **Nota didática sobre Performance:** O `apply(..., axis=1)` é ótimo para aprender lógica linha a linha, mas itera por cada registro em Python puro. No dia a dia em bases médias ou grandes, prefira sempre a **forma vetorizada** do Pandas (executada em C/Rust): `(df['lucro_prod'] / df['vendas_prod']) * 100`.


In [ ]:
df['margem_lucro'] = df.apply(lambda x: (x['lucro_prod'] / x['vendas_prod']) * 100, axis=1)

### 6.4. Nível de vendas e desconto elevado

Em uma coluna, `apply()` passa cada valor para a `lambda`. Vendas acima de 500 recebem 'Alta'; as demais recebem 'Baixa'. Desconto acima de 0.2 (20%) recebe `True`; os demais recebem `False`.

> 💡 **Dica Pro:** Para condições binárias booleanas simples, a comparação direta é vetorizada e muito mais concisa: `df['desconto_concedido'] > 0.2` produz o mesmo resultado sem precisar de `apply` ou `lambda`!


In [ ]:
df['nivel_vendas'] = df['vendas_prod'].apply(lambda x: 'Alta' if x > 500 else 'Baixa')
df['desconto_elevado'] = df['desconto_concedido'].apply(lambda x: True if x > 0.2 else False)

In [ ]:
df[[
    "desconto_aplicado", "lucro_por_unidade", "categoria_lucro",
    "margem_lucro", "nivel_vendas", "desconto_elevado",
]].head()

## 7. Recortes para análise

Depois de preparar a base, crie recortes para responder perguntas específicas. Aqui, comparamos vendas das regiões West e South.


In [ ]:
vendas_por_regiao = (
    df.loc[df["regiao"].isin(["West", "South"])]
    .groupby("regiao", observed=True)["vendas_prod"]
    .sum()
    .round(2)
)

vendas_por_regiao

## Checklist de encerramento

- Os nomes das colunas seguem um padrão?
- Datas, identificadores, categorias e métricas usam tipos coerentes?
- Nulos e duplicidades foram avaliados antes de qualquer remoção?
- Novas variáveis têm uma regra de negócio clara?
- Os resultados finais foram validados contra a base original?

Com esse roteiro, troque apenas o caminho do arquivo, o mapeamento de nomes, os formatos de data e as regras de negocio para reutiliza-lo em uma nova base.
